# Campaign dataset inspection

## tl;dr
The supplied CSV is structurally suitable for our teaching app: 4,040 synthetic rows, 24 campaigns, 183 dates, and USD throughout. No missing cells, duplicate campaign/date keys, invalid values, or gaps within active windows were found. The 352 absent combinations are documented inactivity. These are invented results, not evidence about real advertising performance.


## Context & Methods
Run from `campaign-analyzer` using its Python environment. This notebook inspects the existing CSV without regenerating or overwriting it. It calls `inspect_dataset.py`; independent currency checks use CSV strings and Decimal, while pandas checks structure and grouped arithmetic. `DATA_INSPECTION.json` records detailed results and the source hash.

### Key Assumptions
The supplied guide and generation notebook define one row as one campaign on one date. All conversions are attributed purchases, including Awareness and Traffic goals. Purchases are assigned to one campaign in the simulation. Revenue excludes refunds, product cost, agency fees, and operating costs. Calendar dates have no timezone conversion. Synthetic changes and channel factors are deliberate simulation choices.

Sources: `campaign_data_synthetic.csv`, `campaign_dataset_guide.txt`, and `campaign_dataset_generate_and_check.ipynb`. The `_compatible.csv` is checked for identical values after the two header mappings.


In [1]:
from pathlib import Path
import pandas as pd
from inspect_dataset import run_inspection

assert Path("campaign_data_synthetic.csv").is_file(), "Run from campaign-analyzer"


## Data
A small deterministic preview confirms the row shape. We use campaign ID for grouping and campaign name for display/search. `goal` maps to `campaign_goal` and `dataset_type` maps to `data_source` when supporting the compatible headers.


In [2]:
data = pd.read_csv("campaign_data_synthetic.csv")
print(data.head(3).to_string(index=False))


      date campaign_id                  campaign_name  channel           audience  goal  impressions  clicks  conversions  spend  revenue currency dataset_type
2026-04-01      CMP001 Everyday Essentials - Facebook Facebook        Prospecting Sales         9200      86            2  91.23   219.11      USD    synthetic
2026-04-01      CMP002      Return to Cart - Facebook Facebook        Retargeting Sales         4302      98            2  57.49   217.08      USD    synthetic
2026-04-01      CMP003  Loyalty Collection - Facebook Facebook Existing Customers Sales         6176     145            5  73.68   482.22      USD    synthetic


## Results
### 1. Run the structural and independent calculation checks
The script asserts uniqueness, valid dates and numbers, stable campaign metadata, complete active windows, guide control totals, channel reconciliation, a combined filter, and four zero-denominator cases. Failures stop execution instead of silently discarding rows.


In [3]:
inspection = run_inspection()


PASS: 4,040 synthetic rows, 24 campaigns, 183 dates.
All structural checks passed; no missing days inside active windows.
Baseline: {
  "impressions": 30156909,
  "clicks": 437337,
  "conversions": 7823,
  "spend": "306470.04",
  "revenue": "839690.13",
  "ctr_pct": 1.4502049928260219,
  "cvr_pct": 1.7887807343078679,
  "cpc": 0.70076403322838,
  "cpa": 39.175513230218584,
  "cpm": 10.16251499780697,
  "roas": 2.7398767266124935,
  "ad_spend_return_pct": 173.98767266124938,
  "revenue_per_purchase": 107.33607695257574
}
Valid zero-value rows: {'impressions': 1, 'clicks': 2, 'conversions': 1315, 'spend': 2, 'revenue': 1315}
Combined-filter reference: {
  "date_range": [
    "2026-06-01",
    "2026-06-30"
  ],
  "channel": "Instagram",
  "audience": "Prospecting",
  "goal": "Sales",
  "rows": 60,
  "campaigns": 2,
  "metrics": {
    "impressions": 728034,
    "clicks": 10646,
    "conversions": 175,
    "spend": "9394.52",
    "revenue": "20205.07",
    "ctr_pct": 1.4622943433960502,
   

### 2. Inspect zero denominators
A zero numerator divided by a positive denominator is a valid zero. A zero denominator is unavailable (N/A). The credit-funded day has valid zero CPC/CPA but undefined ROAS, despite positive revenue.


In [4]:
edges = pd.DataFrame(inspection["edge_case_controls"])
print(edges[["date", "campaign_id", "ctr_pct", "cvr_pct", "cpc", "cpa", "roas", "ad_spend_return_pct"]].to_string(index=False, na_rep="N/A"))


      date campaign_id  ctr_pct  cvr_pct      cpc  cpa  roas  ad_spend_return_pct
2026-04-03      CMP001      N/A      N/A      N/A  N/A   N/A                  N/A
2026-06-15      CMP009      0.0      N/A      N/A  N/A   0.0               -100.0
2026-05-09      CMP014      1.4      0.0 0.557143  N/A   0.0               -100.0
2026-09-02      CMP019      1.0     20.0 0.000000  0.0   N/A                  N/A


## Takeaways
- Full-data controls: spend USD 306,470.04; attributed revenue USD 839,690.13; 7,823 purchases; CTR 1.450205%; ROAS 2.739877x.
- June + Instagram + Prospecting + Sales must produce 60 rows and 2 campaigns, USD 9,394.52 spend and USD 20,205.07 revenue.
- Sum raw measures before calculating ratios. Missing days should not be invented as zero activity.
- Supported: totals, CTR, purchase conversion rate, CPC, cost per purchase, CPM, ROAS, ad-spend return, and attributed revenue per purchase. Full profit ROI, unique reach, new-customer CAC, retention, and causal lift are unsupported.
- The uploader now explains invalid essential fields and preserves unknown revenue. Missing revenue must not be filled with zero; revenue-dependent metrics should be unavailable when their selected inputs are incomplete.
- The clean default file does not test malformed-date, duplicate-input, or missing-revenue error handling. Separate loader fixtures now test malformed dates, missing revenue, and duplicate input errors; the clean demo data remains intact.
- CSV loading, validation, and the synthetic-data label are now implemented on port 8502. Next: Overview and shared filters. The numerical results here describe the completed read-only inspection.

Validation: code cells below were executed top-to-bottom in one fresh Python namespace and their actual stdout saved. This is Python-cell execution, not a Jupyter-kernel run; no extra notebook packages were installed.
